# Supervisor, Handoffs, and Multi-Source Routing


In [1]:
# --- Groq API key (free): https://console.groq.com/keys ---
# Add it to Colab Secrets (key icon, left sidebar) as GROQ_API_KEY.
# Never paste the key directly into this cell.
import os
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    pass  # running locally: export GROQ_API_KEY in your shell
assert os.environ.get("GROQ_API_KEY"), "GROQ_API_KEY is not set"
print("Groq key loaded")

Groq key loaded


<a href="https://colab.research.google.com/github/MohammadYusif/agentic-ai-systems/blob/master/notebooks/08b_supervisor_and_handoffs.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

*Run this lesson yourself — opens in Google Colab. You need a free [Groq API key](00b_setup_groq.qmd).*

You have now seen three ways to combine agents, one per lesson. This notebook
puts them side by side, because the capstone asks you to pick one and the
choice is the graded decision.

| Shape | Who decides next | Lesson | Capstone track |
|---|---|---|---|
| **Agents as tools** | The calling agent | [Sub-agents](02_subagents.ipynb) | A |
| **Supervisor + workers** | A dedicated router | this notebook | A |
| **Handoff / escalation** | The agent itself, or a rule | [Customer support](08_customer_support_agent.ipynb) | B |
| **Router across sources** | A classifier | this notebook | C |

All four are built from the same primitive: the constrained-output decision
from [Structured Output as the Routing Primitive](01b_structured_routing.qmd).

In [2]:
%pip install -qU langchain langchain-groq langgraph langgraph-supervisor

In [3]:
%pip install -qU langchain_community

In [4]:
%pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp opentelemetry-exporter-otlp-proto-grpc

Found existing installation: chromadb 0.4.18
Uninstalling chromadb-0.4.18:
  Successfully uninstalled chromadb-0.4.18
Found existing installation: opentelemetry-api 1.42.1
Uninstalling opentelemetry-api-1.42.1:
  Successfully uninstalled opentelemetry-api-1.42.1
Found existing installation: opentelemetry-sdk 1.42.1
Uninstalling opentelemetry-sdk-1.42.1:
  Successfully uninstalled opentelemetry-sdk-1.42.1
Found existing installation: opentelemetry-exporter-otlp-proto-grpc 1.42.1
Uninstalling opentelemetry-exporter-otlp-proto-grpc-1.42.1:
  Successfully uninstalled opentelemetry-exporter-otlp-proto-grpc-1.42.1


In [5]:
# Uninstall potentially conflicting opentelemetry packages (including any left over chromadb) and numpy
%pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp opentelemetry-exporter-otlp-proto-grpc numpy
# Install a known compatible version of chromadb, specific opentelemetry versions, and a compatible numpy version
%pip install -qU chromadb==0.4.18 opentelemetry-api==1.42.1 opentelemetry-sdk==1.42.1 'numpy<2.0.0'

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but

In [6]:
import os
os.environ['CHROMA_SERVER_NO_TELEMETRY'] = '1'

print("ChromaDB telemetry disabled.")

ChromaDB telemetry disabled.


## Shape 1 — Agents as tools (recap)

The pattern from Day 1: a main agent holds sub-agents as tools and calls them
when it decides to. Control always returns to the caller.

**Use it when** the sub-agents are genuinely helpers — the main agent stays in
charge and composes their results.

**Weakness:** the caller carries every sub-agent's description in its own
prompt, so it degrades as you add more.

## Shape 2 — Supervisor + workers

A dedicated agent whose only job is to decide who works next. Workers do not
know about each other.

**Use it when** you have several specialists and want one clear place where
delegation happens (Track A).

In [7]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent # Updated import

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# --- worker tools -------------------------------------------------------
CALENDAR = []


@tool
def create_event(title: str, day: str) -> str:
    """Add an event to the calendar."""
    CALENDAR.append({"title": title, "day": day})
    return f"Scheduled '{title}' on {day}."


@tool
def list_events() -> str:
    """List all calendar events."""
    return str(CALENDAR) if CALENDAR else "No events."


@tool
def draft_email(to: str, subject: str) -> str:
    """Draft (but do not send) an email."""
    return f"Draft to {to} — subject: {subject}"


# --- workers ------------------------------------------------------------
calendar_agent = create_agent(
    model=llm, tools=[create_event, list_events], name="calendar_agent")

email_agent = create_agent(
    model=llm, tools=[draft_email], name="email_agent")

print("workers ready")

workers ready


In [8]:
from langgraph_supervisor import create_supervisor

supervisor = create_supervisor(
    agents=[calendar_agent, email_agent],
    model=llm,
    prompt=(
        "You supervise a calendar assistant and an email assistant. "
        "Route scheduling and availability requests to calendar_agent, and "
        "anything about writing or sending mail to email_agent. "
        "After a worker replies, relay their full answer to the user."
    ),
).compile()

result = supervisor.invoke({"messages": [
    {"role": "user", "content": "Book a design review on Tuesday"}]})

# The handoff is visible as a tool call named transfer_to_<worker>
for m in result["messages"]:
    for tc in getattr(m, "tool_calls", []) or []:
        print("handoff ->", tc["name"])
print()
print(result["messages"][-1].content)

handoff -> transfer_to_calendar_agent
handoff -> transfer_back_to_supervisor

Your design review has been booked for Tuesday.


Expected: a `transfer_to_calendar_agent` call, then the worker's reply.

::: {.callout-tip}
## What full marks looks like
Print the `transfer_to_*` tool calls like the loop above. It is direct evidence
that the **LLM** chose the worker — which is exactly what the multi-agent
section is checking for.
:::

## Shape 3 — Handoff to a human (Track B)

Escalation is a handoff whose recipient is a person. The mechanism is
`interrupt()` — the same one from
[Thinking in LangGraph](09_langgraph.qmd).

The decision of *whether* to escalate should be the model's, not a keyword
rule.

In [9]:
from typing import Literal
from pydantic import BaseModel, Field
from langgraph.func import entrypoint, task
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


class Triage(BaseModel):
    urgency: Literal["low", "medium", "high"] = Field(
        description="high when money, data loss, or an angry customer is involved")
    needs_human: bool = Field(
        description="True when a person must approve before we reply")
    summary: str


triage_llm = llm.with_structured_output(Triage)


@task
def triage(ticket: str) -> Triage:
    return triage_llm.invoke(f"Triage this support ticket:\n\n{ticket}")


@task
def draft_reply(ticket: str, t: Triage) -> str:
    return llm.invoke(
        f"Write a short, professional support reply.\n\nTicket: {ticket}"
    ).content


@entrypoint(checkpointer=InMemorySaver())
def support(ticket: str) -> dict:
    t = triage(ticket).result()
    draft = draft_reply(ticket, t).result()

    if t.needs_human:                       # <- the handoff
        decision = interrupt({
            "action": "Approve or edit before this is sent",
            "urgency": t.urgency,
            "draft": draft,
        })
        if decision != "approve":
            draft = str(decision)           # human rewrote it

    return {"urgency": t.urgency, "escalated": t.needs_human, "reply": draft}

In [10]:
cfg = {"configurable": {"thread_id": "ticket-1"}}

paused = support.invoke("I was charged twice and I want my money back NOW", cfg)
print("PAUSED:", paused["__interrupt__"][0].value["urgency"])
print("draft:", paused["__interrupt__"][0].value["draft"][:70], "...")

done = support.invoke(
    Command(resume="We've refunded the duplicate charge — it lands in 5 days."),
    cfg)
print("\nFINAL:", done)

PAUSED: high
draft: Dear valued customer,

I apologize for the inconvenience you've experi ...

FINAL: {'urgency': 'high', 'escalated': True, 'reply': "We've refunded the duplicate charge — it lands in 5 days."}


Note the human's edit appears in the final reply — that is the difference
between demonstrating a handoff and merely pausing.

::: {.callout-important}
A workflow that stops at the interrupt has shown half of Track B. Always run
the `Command(resume=...)` and keep its output.
:::

## Shape 4 — Router across sources (Track C)

Two knowledge bases, one classifier deciding which to search. The mistake to
avoid is building two retrievers and then querying **the same store** for both
— the routing then changes nothing.

In [16]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_community.embeddings.fake import FakeEmbeddings
from langgraph.func import task # Import task here if not already imported

# Define placeholders for missing variables
academic_chunks = [
    Document(page_content="The history of artificial intelligence in academia."),
    Document(page_content="Research on neural networks at major universities.")
]
campus_chunks = [
    Document(page_content="Information about student dormitories and dining halls."),
    Document(page_content="Guidelines for campus events and activities.")
]
embeddings = FakeEmbeddings(size=1536) # Dummy embeddings model for demonstration

# Two SEPARATE stores. This is what makes routing meaningful.
academic_store = Chroma.from_documents(academic_chunks, embeddings,
                                       collection_name="academic")
campus_store   = Chroma.from_documents(campus_chunks,   embeddings,
                                       collection_name="campus")

# Define retrievers from the stores, setting k=2 to avoid the warning
academic_retriever = academic_store.as_retriever(search_kwargs={"k": 2})
campus_retriever = campus_store.as_retriever(search_kwargs={"k": 2})

@task
def retrieve(question: str, destination: str) -> str:
    # The return type annotation for retrieve is `str`, but
    # academic_retriever.invoke() returns a list of Document objects.
    # We need to extract the content and combine it into a string.
    if destination == "academic":
        docs = academic_retriever.invoke(question)
        return "Academic: " + " ".join([doc.page_content for doc in docs])
    elif destination == "campus":
        docs = campus_retriever.invoke(question)
        return "Campus: " + " ".join([doc.page_content for doc in docs])
    else:                                    # "both"
        academic_docs = academic_retriever.invoke(question)
        campus_docs = campus_retriever.invoke(question)
        return (
            "Academic: " + " ".join([doc.page_content for doc in academic_docs]) +
            "\nCampus: " + " ".join([doc.page_content for doc in campus_docs])
        )

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [14]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_community.embeddings.fake import FakeEmbeddings

# Define placeholders for missing variables
academic_chunks = [
    Document(page_content="The history of artificial intelligence in academia."),
    Document(page_content="Research on neural networks at major universities.")
]
campus_chunks = [
    Document(page_content="Information about student dormitories and dining halls."),
    Document(page_content="Guidelines for campus events and activities.")
]
embeddings = FakeEmbeddings(size=1536) # Dummy embeddings model for demonstration

# Two SEPARATE stores. This is what makes routing meaningful.
academic_store = Chroma.from_documents(academic_chunks, embeddings,
                                       collection_name="academic")
campus_store   = Chroma.from_documents(campus_chunks,   embeddings,
                                       collection_name="campus")

# Define retrievers from the stores, setting k=2 to avoid the warning
academic_retriever = academic_store.as_retriever(search_kwargs={"k": 2})
campus_retriever = campus_store.as_retriever(search_kwargs={"k": 2})

@task
def retrieve(question: str, destination: str) -> str:
    # The return type annotation for retrieve is `str`, but
    # academic_retriever.invoke() returns a list of Document objects.
    # We need to extract the content and combine it into a string.
    if destination == "academic":
        docs = academic_retriever.invoke(question)
        return "Academic: " + " ".join([doc.page_content for doc in docs])
    elif destination == "campus":
        docs = campus_retriever.invoke(question)
        return "Campus: " + " ".join([doc.page_content for doc in docs])
    else:                                    # "both"
        academic_docs = academic_retriever.invoke(question)
        campus_docs = campus_retriever.invoke(question)
        return (
            "Academic: " + " ".join([doc.page_content for doc in academic_docs]) +
            "\nCampus: " + " ".join([doc.page_content for doc in campus_docs])
        )

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Now that the `Chroma` stores and `retrieve` function are set up, let's test the `retrieve` function with some sample questions to see if it fetches the correct information.

In [15]:
from langgraph.func import entrypoint
from typing import Dict, Any

@entrypoint()
def test_retrieve_flow(input_data: Dict[str, Any]):
    query = input_data["query"]
    destination = input_data["destination"]
    return retrieve(query, destination).result()

print('--- Testing Academic Query ---')
academic_query = "What is the history of AI research?"
academic_result = test_retrieve_flow.invoke({"query": academic_query, "destination": "academic"})
print(academic_result)

print('\n--- Testing Campus Query ---')
campus_query = "Where can I find information about student dormitories?"
campus_result = test_retrieve_flow.invoke({"query": campus_query, "destination": "campus"})
print(campus_result)

print('\n--- Testing Both Sources Query ---')
both_query = "Tell me about AI research and campus activities."
both_result = test_retrieve_flow.invoke({"query": both_query, "destination": "both"})
print(both_result)

--- Testing Academic Query ---
Academic: The history of artificial intelligence in academia. Research on neural networks at major universities.

--- Testing Campus Query ---
Campus: Guidelines for campus events and activities. Guidelines for campus events and activities.

--- Testing Both Sources Query ---
Academic: Research on neural networks at major universities. Research on neural networks at major universities.
Campus: Guidelines for campus events and activities. Information about student dormitories and dining halls.


Pair it with the `MultiRoute` classifier from
[the routing lesson](01b_structured_routing.qmd).

**The test that proves it works:** ask a question answerable only from source A
and confirm the trace shows only source A was searched — then do the same for
B.

## Choosing

| If your problem is... | Use |
|---|---|
| One assistant that occasionally needs a specialist | Agents as tools |
| Several specialists, one clear delegation point | Supervisor + workers |
| Work that must reach a person before it is final | Handoff / `interrupt()` |
| One question, several possible knowledge sources | Router across sources |

Pick the one that matches your capstone track, name it in your write-up, and
show the evidence: printed `transfer_to_*` calls for a supervisor, a completed
interrupt/resume for a handoff, or per-source retrieval for a router.